In [25]:
import os
import re
import pandas as pd

RAVDESS_PATH = "data/ravdess"
SAVEE_PATH = "data/savee"

# Emotion -> Label
emotion_to_label = {
    "neutral": 0,
    "calm": 1,
    "happy": 2,
    "sad": 3,
    "angry": 4,
    "fear": 5,
    "disgust": 6,
    "surprise": 7
}

# Emotion code trong tên file
code_to_emotion = {
    "a": "angry",
    "d": "disgust",
    "f": "fear",
    "h": "happy",
    "n": "neutral",
    "sa": "sad",
    "su": "surprise"
}

emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fear",
    "07": "disgust",
    "08": "surprise"
}

In [26]:
records = []

for actor in os.listdir(RAVDESS_PATH):

    for file_name in os.listdir(f"{RAVDESS_PATH}/{actor}"):

        if not file_name.endswith(".wav"):
            continue

        file_path = f"{RAVDESS_PATH}/{actor}/{file_name}"

        parts = file_name.replace(".wav", "").split("-")

        emotion_code = parts[2]
        speaker = parts[6]

        emotion = emotion_map[emotion_code]
        label = emotion_to_label[emotion]

        records.append({
            "path": file_path,
            "speaker": speaker,
            "emotion": emotion,
            "label": label,
            "dataset": "RAVDESS"
        })

ravdess_df = pd.DataFrame(records)

print(ravdess_df.head())
print(ravdess_df["emotion"].value_counts())
print("Total:", len(ravdess_df))

os.makedirs("CSVs", exist_ok=True)
ravdess_df.to_csv("CSVs/ravdess_metadata.csv", index=False)

print("\nSaved to CSVs/ravdess_metadata.csv")

                                             path speaker  emotion  label  \
0  data/ravdess/Actor_01/03-01-01-01-01-01-01.wav      01  neutral      0   
1  data/ravdess/Actor_01/03-01-01-01-01-02-01.wav      01  neutral      0   
2  data/ravdess/Actor_01/03-01-01-01-02-01-01.wav      01  neutral      0   
3  data/ravdess/Actor_01/03-01-01-01-02-02-01.wav      01  neutral      0   
4  data/ravdess/Actor_01/03-01-02-01-01-01-01.wav      01     calm      1   

   dataset  
0  RAVDESS  
1  RAVDESS  
2  RAVDESS  
3  RAVDESS  
4  RAVDESS  
emotion
calm        192
happy       192
sad         192
angry       192
fear        192
disgust     192
surprise    192
neutral      96
Name: count, dtype: int64
Total: 1440

Saved to CSVs/ravdess_metadata.csv


In [27]:
records = []

for file_name in os.listdir(SAVEE_PATH):

    if not file_name.lower().endswith(".wav"):
        continue

    file_path = f"{SAVEE_PATH}/{file_name}"

    # Ví dụ:
    # DC_a01.wav
    # DC_sa05.wav
    # JE_su03.wav

    base_name = os.path.splitext(file_name)[0]

    speaker, emotion_part = base_name.split("_")

    # Lấy phần chữ trước số:
    # a01  -> a
    # sa05 -> sa
    # su03 -> su
    emotion_code = re.match(r"[a-z]+", emotion_part).group() # type: ignore

    emotion = code_to_emotion[emotion_code]
    label = emotion_to_label[emotion]

    records.append({
        "path": file_path,
        "speaker": speaker,
        "emotion": emotion,
        "label": label,
        "dataset": "SAVEE"
    })

# Tạo DataFrame
savee_df = pd.DataFrame(records)

# Sắp xếp cho đẹp
savee_df = savee_df.sort_values("path").reset_index(drop=True)

print(savee_df.head())
print()
print("Total samples:", len(savee_df))
print()
print(savee_df["emotion"].value_counts())
os.makedirs("CSVs", exist_ok=True)

savee_df.to_csv("CSVs/savee_metadata.csv", index=False)

print("\nSaved to CSVs/savee_metadata.csv")

                    path speaker emotion  label dataset
0  data/savee/DC_a01.wav      DC   angry      4   SAVEE
1  data/savee/DC_a02.wav      DC   angry      4   SAVEE
2  data/savee/DC_a03.wav      DC   angry      4   SAVEE
3  data/savee/DC_a04.wav      DC   angry      4   SAVEE
4  data/savee/DC_a05.wav      DC   angry      4   SAVEE

Total samples: 480

emotion
neutral     120
angry        60
disgust      60
fear         60
happy        60
sad          60
surprise     60
Name: count, dtype: int64

Saved to CSVs/savee_metadata.csv


In [ ]:
combined_df = pd.concat(
    [savee_df, ravdess_df],
    ignore_index=True
)

os.makedirs("CSVs", exist_ok=True)
combined_df.to_csv("CSVs/combined_metadata.csv", index=False)

print(combined_df.shape)
print(combined_df.head())

(1920, 5)
                    path speaker emotion  label dataset
0  data/savee/DC_a01.wav      DC   angry      4   SAVEE
1  data/savee/DC_a02.wav      DC   angry      4   SAVEE
2  data/savee/DC_a03.wav      DC   angry      4   SAVEE
3  data/savee/DC_a04.wav      DC   angry      4   SAVEE
4  data/savee/DC_a05.wav      DC   angry      4   SAVEE
